In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir()
)
sys.path.insert(0, str(project_root))

from src.data.loader import DatasetLoader
from src.utils.config import Config

# DATASET = "CIC-IoT2023"
DATASET = "MITS-Network"
loader = DatasetLoader(Config())
df = loader.load(DATASET, sample_size=1000)

print(f"Dataset: {DATASET}")
print(f"Shape: {df.shape}")

Dataset: MITS-Network
Shape: (1000, 26)


In [2]:
from pathlib import Path
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir()
)
sys.path.insert(0, str(project_root))

from src.evaluation.dss import DatasetDSSAnalyzer

analyzer = DatasetDSSAnalyzer(
    df=df,
    dataset_name=DATASET
)

dss = analyzer.calculate()

print(f"DSS for {DATASET}: {dss:.3f} / 5.000")
print(f"DSS percentage: {(dss / 5) * 100:.2f}%")

DSS for MITS-Network: 2.650 / 5.000
DSS percentage: 53.00%


In [3]:
result = analyzer.detailed_result()

pd.DataFrame([result])

,Dataset,Alert/Event,Context,Threat Intelligence,Exploitability,Historical,Ground Truth,Scalability,DSS,DSS (%),Analyzed Rows,Total Rows
0,MITS-Network,4.0,4.5,1.0,1.0,4.0,0.0,1.0,2.65,53.0,1000,None


In [4]:
evidence_df = analyzer.evidence_table()

display(evidence_df)

,Dataset,Criterion,Score,Weight,Weighted Score,Evidence
0,MITS-Network,Alert/Event,4.0,0.20,0.80,Alert/event fields: ['event_type'] | Temporal ...
1,MITS-Network,Context,4.5,0.20,0.90,"Native context fields: ['src_hostname', 'dst_h..."
2,MITS-Network,Threat Intelligence,1.0,0.15,0.15,External TI enrichment potential via network i...
3,MITS-Network,Exploitability,1.0,0.10,0.10,Asset mapping permits external CVE/CVSS/exploi...
4,MITS-Network,Historical,4.0,0.15,0.60,"Temporal fields: ['timestamp', 'duration_ms'] ..."
5,MITS-Network,Ground Truth,0.0,0.10,0.00,No ground-truth/label field detected
6,MITS-Network,Scalability,1.0,0.10,0.10,Full dataset size unavailable; analyzed sample...


In [5]:
# Discover non-empty dataset directories using their actual folder names.
DATASETS = sorted(
    path.name
    for path in Config.DATA_ROOT.iterdir()
    if path.is_dir()
    and any(
        file.is_file() and file.suffix.lower() in {".csv", ".parquet"}
        for file in path.rglob("*")
    )
)

print(f"Available datasets: {DATASETS}")

Available datasets: ['BCCC-CIC-IDS-2017', 'BCCC-CSE-IDS2018', 'BigFlow-Parquet', 'CIC-IoT2023', 'MITS-Network', 'TON-IoT', 'UNSW-NB15']


In [6]:
all_results = []
all_evidence = []

for dataset_name in DATASETS:

    print("=" * 70)
    print(f"Processing: {dataset_name}")

    try:
        # Development/testing sample: keeps memory use manageable.
        df = loader.load(dataset_name, sample_size=10000)

        # IMPORTANT: obtain full-dataset metadata separately so that the
        # scalability criterion is not distorted by the 10k-row sample.
        metadata = loader.metadata(dataset_name)

        print(f"Shape analyzed: {df.shape}")
        print(f"Full dataset rows: {metadata['total_rows']:,}"
              if metadata["total_rows"] is not None
              else "Full dataset rows: unavailable")

        analyzer = DatasetDSSAnalyzer(
            df=df,
            dataset_name=dataset_name,
            total_rows=metadata["total_rows"],
            source_files=metadata["files"],
            sample_size=10000,
        )

        result = analyzer.detailed_result()
        all_results.append(result)

        evidence = analyzer.evidence_table()
        all_evidence.append(evidence)

        print(
            f"DSS = {result['DSS']:.3f}/5 "
            f"({result['DSS (%)']:.2f}%)"
        )

    except Exception as error:
        print(f"ERROR processing {dataset_name}: {error}")


Processing: BCCC-CIC-IDS-2017
Shape analyzed: (10000, 122)
Full dataset rows: 2,438,052
DSS = 2.750/5 (55.00%)
Processing: BCCC-CSE-IDS2018
Shape analyzed: (10000, 80)
Full dataset rows: 16,233,002
DSS = 1.750/5 (35.00%)
Processing: BigFlow-Parquet
Shape analyzed: (10000, 55)
Full dataset rows: 46,000,000
DSS = 2.950/5 (59.00%)
Processing: CIC-IoT2023
Shape analyzed: (10000, 47)
Full dataset rows: 46,686,579
DSS = 2.450/5 (49.00%)
Processing: MITS-Network
Shape analyzed: (2590, 26)
Full dataset rows: 2,590
DSS = 2.650/5 (53.00%)
Processing: TON-IoT
Shape analyzed: (10000, 6)
Full dataset rows: 31,864,826
DSS = 1.600/5 (32.00%)
Processing: UNSW-NB15
Shape analyzed: (10000, 49)
Full dataset rows: 2,540,251
DSS = 0.400/5 (8.00%)


In [7]:
dss_table = pd.DataFrame(all_results)

evidence_table = pd.concat(
    all_evidence,
    ignore_index=True
)

In [8]:
dss_ranking = (
    dss_table
    .sort_values(by="DSS", ascending=False)
    .reset_index(drop=True)
)

dss_ranking.insert(0, "Rank", range(1, len(dss_ranking) + 1))

# Keep only the paper-facing DSS matrix.
paper_columns = [
    "Rank", "Dataset", "Alert/Event", "Context",
    "Threat Intelligence", "Exploitability", "Historical",
    "Ground Truth", "Scalability", "DSS", "DSS (%)",
]
display(dss_ranking[paper_columns])


,Rank,Dataset,Alert/Event,Context,Threat Intelligence,Exploitability,Historical,Ground Truth,Scalability,DSS,DSS (%)
0,1,BigFlow-Parquet,5.0,1.0,1.0,1.0,4.0,4.0,5.0,2.95,59.0
1,2,BCCC-CIC-IDS-2017,3.0,2.5,1.0,1.0,4.0,4.0,4.0,2.75,55.0
2,3,MITS-Network,4.0,4.5,1.0,1.0,4.0,0.0,1.0,2.65,53.0
3,4,CIC-IoT2023,2.0,0.0,4.0,0.0,3.0,5.0,5.0,2.45,49.0
4,5,BCCC-CSE-IDS2018,2.0,0.0,0.0,0.0,3.0,4.0,5.0,1.75,35.0
5,6,TON-IoT,2.0,0.0,0.0,0.0,2.0,4.0,5.0,1.60,32.0
6,7,UNSW-NB15,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.40,8.0
